# Notebook 07 — Robustness: sensitivity to fitting range

Tests whether the regional aperiodic patterns are stable across alternative  
specparam fitting frequency ranges (Section 2.3.6).

**Main question**: Does the rank order of regions by exponent remain consistent  
when we change the fitting range from the default `(2, 40)` Hz?

**Ranges tested**:
- `(1, 40)` — extend down to 1 Hz (more delta)
- `(2, 40)` — default (main analysis)
- `(3, 40)` — match original Frauscher/fooof analysis
- `(2, 30)` — exclude high gamma
- `(2, 80)` — extend to broadband (if data quality permits)

**Input**: Raw PSDs from iEEG (recomputed as in NB 02)

**Output**: Supplementary robustness figure (Section 2.3.6 / appendix)

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import scipy.io
import h5py
from scipy.signal import welch
from scipy.stats import spearmanr
import matplotlib.pyplot as plt
from specparam import SpectralGroupModel

PROJECT_ROOT = Path("../../").resolve()
INTERIM_DIR  = PROJECT_ROOT / "data" / "interim"
FIG_DIR      = PROJECT_ROOT / "manuscript" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

R2_THRESHOLD      = 0.80
APERIODIC_MODE    = "fixed"
PEAK_WIDTH_LIMITS = [1.0, 12.0]
MAX_N_PEAKS       = 6
MIN_PEAK_HEIGHT   = 0.05
PEAK_THRESHOLD    = 2.0
FS = 200.0

# Fitting ranges to compare
FREQ_RANGES = [
    (1.0,  40.0),
    (2.0,  40.0),  # default
    (3.0,  40.0),
    (2.0,  30.0),
    (2.0,  80.0),
]

## 1. Load iEEG data

In [ ]:
IEEG_MAT   = PROJECT_ROOT / "data" / "Frauscher2018" / "WakefulnessMatlabFile.mat"
REGION_INFO = PROJECT_ROOT / "data" / "Frauscher2018" / "RegionInformation.csv"

print("Loading iEEG data …")
try:
    mat = scipy.io.loadmat(str(IEEG_MAT))
    data       = mat["Data"].T
    ch_regions = np.array(mat["ChannelRegion"]).flatten().astype(int)
except Exception:
    with h5py.File(IEEG_MAT, "r") as f:
        data       = np.array(f["Data"]).T
        ch_regions = np.array(f["ChannelRegion"]).flatten().astype(int)

freqs, psds = welch(
    data, fs=FS, nperseg=int(2*FS), noverlap=int(FS),
    detrend="constant", scaling="density"
)

region_info = pd.read_csv(REGION_INFO)
region_info.columns = region_info.columns.str.strip()
for col in region_info.select_dtypes("object").columns:
    region_info[col] = region_info[col].str.strip("'")

print(f"PSDs: {psds.shape}")

## 2. Fit specparam for each frequency range

In [ ]:
def specparam2pandas(fg):
    """
    Converts a SpectralGroupModel object into a pandas DataFrame, with peak parameters and
    corresponding aperiodic fit information.

    Returns a long-format DataFrame where each row represents a single peak.
    Spectra with no peaks are retained with NaN values for peak columns.
    Columns: CF, PW, BW, ID, offset, exponent (+ knee if knee mode), error_mae, gof_rsquared.
    """
    if not fg.results.has_model:
        raise ValueError("No model fit results available. Please fit the model first.")

    ap_params = fg.get_params("aperiodic")
    ap_labels = list(fg.modes.aperiodic.params.labels)

    specparam_aperiodic = pd.DataFrame(ap_params, columns=ap_labels)

    # Direct dict access — fg.get_metrics() silently returns NaN in some rc6 builds
    group_res = fg.results.group_results
    specparam_aperiodic["error_mae"]    = [r.metrics.get("error_mae",    np.nan) for r in group_res]
    specparam_aperiodic["gof_rsquared"] = [r.metrics.get("gof_rsquared", np.nan) for r in group_res]
    specparam_aperiodic = specparam_aperiodic.reset_index(names=["ID"])

    peaks = fg.get_params("peak")

    if peaks.size > 0:
        peak_df = pd.DataFrame(peaks, columns=["CF", "PW", "BW", "ID"])
        peak_df["ID"] = peak_df["ID"].astype(int)
        result = specparam_aperiodic.merge(peak_df, on="ID", how="left")
    else:
        peak_df = pd.DataFrame(columns=["CF", "PW", "BW", "ID"])
        result = specparam_aperiodic.merge(peak_df, on="ID", how="left")

    return result


all_results = []

for freq_range in FREQ_RANGES:
    label = f"{freq_range[0]:.0f}–{freq_range[1]:.0f} Hz"
    mask  = (freqs >= freq_range[0]) & (freqs <= freq_range[1])
    f_fit = freqs[mask]
    p_fit = psds[:, mask]

    fg = SpectralGroupModel(
        peak_width_limits = PEAK_WIDTH_LIMITS,
        max_n_peaks       = MAX_N_PEAKS,
        min_peak_height   = MIN_PEAK_HEIGHT,
        peak_threshold    = PEAK_THRESHOLD,
        aperiodic_mode    = APERIODIC_MODE,
    )
    fg.fit(f_fit, p_fit, n_jobs=-1)

    # Use aperiodic-only summary for robustness comparison (one row per channel)
    ch_df = specparam2pandas(fg).drop_duplicates(subset="ID")[["ID", "offset", "exponent", "error_mae", "gof_rsquared"]].copy()
    ch_df["freq_range"] = label
    ch_df["region"]     = ch_regions
    ch_df["good_fit"]   = ch_df["gof_rsquared"] >= R2_THRESHOLD
    all_results.append(ch_df)
    print(f"  {label}: done  (good fits: {ch_df['good_fit'].sum()}/{len(ch_df)})")

results_long = pd.concat(all_results, ignore_index=True)
print(f"\nTotal rows: {len(results_long)}")

## 3. Regional exponent across fitting ranges

In [ ]:
# Region-level median per fitting range
def region_agg(df):
    good = df[df["good_fit"] == True]
    return (
        good.groupby(["freq_range", "region"])
        .agg(exponent_median=("exponent", "median"),
             exponent_n     =("exponent", "count"))
        .reset_index()
    )

region_df = region_agg(results_long)

# Merge region names
region_df = region_df.merge(
    region_info.rename(columns={"Region": "region"}),
    on="region", how="left"
)
print(region_df.head(5))

## 4. Rank-order stability: Spearman correlations

In [ ]:
# Pivot to wide format: rows = regions, columns = freq ranges
pivot = region_df.pivot_table(
    index="region", columns="freq_range", values="exponent_median"
)

default_label = "2–40 Hz"
print(f"Spearman correlation of each range vs. default ({default_label}):")
for col in pivot.columns:
    if col == default_label:
        continue
    mask = pivot[[default_label, col]].notna().all(axis=1)
    rho, p = spearmanr(pivot.loc[mask, default_label], pivot.loc[mask, col])
    print(f"  {col:>12s}: ρ = {rho:.3f}  (p = {p:.4f})")

## 5. Robustness figure

In [ ]:
LOBE_COLORS = {
    "Frontal":   "#E64B35",
    "Temporal":  "#4DBBD5",
    "Parietal":  "#00A087",
    "Occipital": "#3C5488",
    "Limbic":    "#F39B7F",
    "Insula":    "#8491B4",
    "Other":     "#91D1C2",
}

n_ranges = len(FREQ_RANGES)
fig, axes = plt.subplots(1, n_ranges - 1, figsize=(5 * (n_ranges - 1), 6), sharey=False)

default_label = "2–40 Hz"
other_labels  = [f"{fr[0]:.0f}–{fr[1]:.0f} Hz" for fr in FREQ_RANGES if fr != (2.0, 40.0)]

for ax, alt_label in zip(axes, other_labels):
    sub = pivot[[default_label, alt_label]].dropna()
    # Attach lobe for colouring
    lobe_map = region_df[["region", "Lobe"]].drop_duplicates().set_index("region")["Lobe"]
    colors = [LOBE_COLORS.get(str(lobe_map.get(r, "Other")), LOBE_COLORS["Other"]) for r in sub.index]

    ax.scatter(sub[default_label], sub[alt_label], c=colors, s=50, alpha=0.8, edgecolors="white")
    lims = [sub.min().min() - 0.05, sub.max().max() + 0.05]
    ax.plot(lims, lims, "k--", alpha=0.4)
    ax.set_xlabel(f"Exponent ({default_label})")
    ax.set_ylabel(f"Exponent ({alt_label})")

    rho, p = spearmanr(sub[default_label], sub[alt_label])
    ax.set_title(f"{alt_label}\nρ = {rho:.2f}")

legend_patches = [
    plt.Rectangle((0, 0), 1, 1, fc=c, label=l)
    for l, c in LOBE_COLORS.items()
]
axes[-1].legend(handles=legend_patches, fontsize=7, loc="upper left")

fig.suptitle("Robustness — Regional exponent across fitting ranges", fontsize=12)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig_robustness_fitting_range.svg", bbox_inches="tight", dpi=150)
plt.show()

## 6. Summary table: exponent stability by region

In [ ]:
# Coefficient of variation across fitting ranges, per region
pivot_named = pivot.copy()
pivot_named["region_name"] = [
    region_info.set_index("Region").loc[r, "Region name"] if r in region_info["Region"].values else f"Region {r}"
    for r in pivot_named.index
]
pivot_named["CV"] = pivot.std(axis=1) / pivot.mean(axis=1)
pivot_named = pivot_named.sort_values("CV")

print("Most stable regions (low CV across fitting ranges):")
display(pivot_named.head(10).round(3))

print("\nLeast stable regions (high CV):")
display(pivot_named.tail(10).round(3))

# Save
results_long.to_csv(INTERIM_DIR / "specparam_ieeg_robustness_freq_ranges.csv", index=False)
pivot_named.to_csv(INTERIM_DIR / "region_exponent_by_freq_range.csv")
print("\nSaved robustness results.")